# Demo Video RAG chỉ dùng PhoBERT

Notebook demo này rút gọn từ `vrag-done.ipynb`:

1. Nhập link YouTube.
2. Lấy metadata và phụ đề tiếng Việt.
3. Chia transcript thành block 60 giây.
4. Dùng `vinai/phobert-base` để tạo embedding.
5. Tính độ tương đồng giữa các block và gộp thành topic.
6. Hiển thị block/topic cho xem.
7. Tóm tắt topic và toàn video bằng Qwen.
8. Nhập câu hỏi và hỏi đáp theo RAG.


In [ ]:
# Nếu chạy trên Kaggle/Colab, chạy cell này trước.
!pip install -q -U "transformers" "accelerate" "bitsandbytes>=0.46.1" "sentencepiece" yt-dlp youtube-transcript-api pandas matplotlib

In [ ]:
import os
import re
import html
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import yt_dlp

from IPython.display import display, Markdown
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, BitsAndBytesConfig
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import NoTranscriptFound, TranscriptsDisabled, VideoUnavailable

BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "demo_data"
OUTPUT_DIR = BASE_DIR / "demo_outputs"
SUB_DIR = DATA_DIR / "subtitles"
META_DIR = DATA_DIR / "metadata"

for p in [DATA_DIR, OUTPUT_DIR, SUB_DIR, META_DIR]:
    p.mkdir(parents=True, exist_ok=True)

PHOBERT_MODEL = "vinai/phobert-base"
SUMMARY_MODEL = "Qwen/Qwen2.5-7B-Instruct"
QA_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
BLOCK_SIZE = 60
TOPIC_SHIFT_THRESHOLD = 0.90
TOP_K = 5

device = "cuda" if torch.cuda.is_available() else "cpu"
print("CUDA available:", torch.cuda.is_available())
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Nhập link YouTube

In [ ]:
youtube_url = input("Nhập link YouTube: ").strip()
if not youtube_url:
    youtube_url = "https://youtu.be/ut7-CUa3f9s?si=nQtCxoCmO_j-aaPs"
print("YouTube URL:", youtube_url)

## 2. Hàm lấy metadata và phụ đề

In [ ]:
def get_youtube_metadata(url: str) -> dict:
    ydl_opts = {
        "quiet": True,
        "skip_download": True,
        "ignoreerrors": True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=False)
    if not info:
        raise ValueError("Không lấy được metadata video.")

    metadata = {
        "video_id": info.get("id"),
        "youtube_url": url,
        "title": info.get("title"),
        "description": info.get("description"),
        "tags": info.get("tags", []),
        "total_duration": info.get("duration"),
        "channel_name": info.get("channel"),
        "channel_id": info.get("channel_id"),
        "uploader": info.get("uploader"),
        "view_count": info.get("view_count"),
        "like_count": info.get("like_count"),
        "comment_count": info.get("comment_count"),
        "published_at": info.get("upload_date"),
        "thumbnail": info.get("thumbnail"),
        "webpage_url": info.get("webpage_url"),
    }

    if metadata["published_at"] and len(metadata["published_at"]) == 8:
        d = metadata["published_at"]
        metadata["published_at"] = f"{d[:4]}-{d[4:6]}-{d[6:]}"
    return metadata


def clean_segment_text(text: str) -> str:
    if text is None:
        return ""
    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", " ", text)
    noise_patterns = [
        r"\[music\]", r"\[applause\]", r"\[laughter\]", r"\[sound\]", r"\[noise\]",
        r"\(music\)", r"\(applause\)", r"\(laughter\)",
    ]
    for pattern in noise_patterns:
        text = re.sub(pattern, " ", text, flags=re.IGNORECASE)
    text = text.replace("\n", " ")
    return re.sub(r"\s+", " ", text).strip()


def normalize_transcript(items):
    segments = []
    for item in items:
        if isinstance(item, dict):
            start = item.get("start", 0)
            duration = item.get("duration", 0)
            text = item.get("text", "")
        else:
            start = getattr(item, "start", 0)
            duration = getattr(item, "duration", 0)
            text = getattr(item, "text", "")

        start = float(start)
        duration = float(duration)
        end = start + duration
        text = clean_segment_text(text)
        if text:
            segments.append({
                "start": round(start, 3),
                "duration": round(duration, 3),
                "end": round(end, 3),
                "text": text,
            })
    return segments


def download_vi_subtitle(video_id: str):
    api = YouTubeTranscriptApi()
    try:
        transcript_list = api.list(video_id)
        print("Danh sách phụ đề có sẵn:")
        for transcript in transcript_list:
            print("-", transcript.language_code, "|", transcript.language, "| generated:", transcript.is_generated)

        vi_transcript = transcript_list.find_transcript(["vi"])
        return normalize_transcript(vi_transcript.fetch())

    except NoTranscriptFound:
        print("Không tìm thấy phụ đề tiếng Việt cho video này.")
    except TranscriptsDisabled:
        print("Video này đã tắt phụ đề.")
    except VideoUnavailable:
        print("Video không khả dụng.")
    except Exception as e:
        print("Lỗi khi tải phụ đề:", e)
    return None

In [ ]:
metadata = get_youtube_metadata(youtube_url)
video_id = metadata["video_id"]

metadata_path = META_DIR / f"{video_id}_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

segments = download_vi_subtitle(video_id)
if not segments:
    raise RuntimeError("Không có phụ đề tiếng Việt. Hãy chọn video khác hoặc thêm Whisper fallback.")

subtitle_data = {
    "video_id": video_id,
    "language": "vi",
    "total_segments": len(segments),
    "segments": segments,
}
subtitle_path = SUB_DIR / f"{video_id}_vi.json"
with open(subtitle_path, "w", encoding="utf-8") as f:
    json.dump(subtitle_data, f, ensure_ascii=False, indent=2)

print("Video ID:", video_id)
print("Title:", metadata["title"])
print("Saved metadata:", metadata_path)
print("Saved subtitle:", subtitle_path)
print("Total subtitle segments:", len(segments))

## 3. Chia transcript thành block 60 giây

In [ ]:
def format_time(seconds):
    seconds = int(seconds)
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    if h > 0:
        return f"{h:02d}:{m:02d}:{s:02d}"
    return f"{m:02d}:{s:02d}"


def split_transcript_to_blocks(transcript, block_size=60):
    blocks = []
    current_block = None

    for item in transcript:
        start = float(item["start"])
        end = float(item.get("end", start + float(item.get("duration", 0))))
        text = item.get("text", "").strip()
        if not text:
            continue

        block_id = int(start // block_size)
        if current_block is None:
            current_block = {
                "block_id": block_id,
                "start_time": block_id * block_size,
                "end_time": (block_id + 1) * block_size,
                "texts": [],
            }

        if block_id != current_block["block_id"]:
            current_block["text"] = " ".join(current_block["texts"])
            del current_block["texts"]
            blocks.append(current_block)

            current_block = {
                "block_id": block_id,
                "start_time": block_id * block_size,
                "end_time": (block_id + 1) * block_size,
                "texts": [],
            }

        current_block["texts"].append(text)

    if current_block is not None and current_block["texts"]:
        current_block["text"] = " ".join(current_block["texts"])
        del current_block["texts"]
        blocks.append(current_block)

    return blocks


blocks = split_transcript_to_blocks(segments, block_size=BLOCK_SIZE)
print("Total blocks:", len(blocks))

In [ ]:
blocks_preview = pd.DataFrame([
    {
        "block_id": b["block_id"],
        "timestamp": f'{format_time(b["start_time"])} - {format_time(b["end_time"])}',
        "text_preview": b["text"][:300],
        "chars": len(b["text"]),
    }
    for b in blocks
])
display(blocks_preview)

with open(OUTPUT_DIR / "blocks_raw.json", "w", encoding="utf-8") as f:
    json.dump(blocks, f, ensure_ascii=False, indent=2)

## 4. Load PhoBERT và tạo embedding cho block

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(PHOBERT_MODEL)
phobert = AutoModel.from_pretrained(PHOBERT_MODEL).to(device)
phobert.eval()
print("Loaded:", PHOBERT_MODEL)

In [ ]:
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
    sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
    return sum_embeddings / sum_mask


@torch.no_grad()
def get_phobert_embedding(text, max_length=256):
    inputs = tokenizer(
        text,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = phobert(**inputs)
    embedding = mean_pooling(outputs, inputs["attention_mask"])
    embedding = F.normalize(embedding, p=2, dim=1)
    return embedding.cpu().numpy()[0]


def embed_blocks_with_phobert(blocks):
    for i, block in enumerate(blocks):
        block["embedding_model"] = PHOBERT_MODEL
        block["embedding"] = get_phobert_embedding(block["text"]).tolist()
        if (i + 1) % 5 == 0 or i == len(blocks) - 1:
            print(f"Embedded {i + 1}/{len(blocks)} blocks")
    return blocks


blocks = embed_blocks_with_phobert(blocks)

## 5. Tính độ tương đồng và gộp block thành topic

In [ ]:
def cosine_similarity(vec_a, vec_b):
    a = np.array(vec_a)
    b = np.array(vec_b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))


def compute_adjacent_similarity(blocks):
    for i in range(len(blocks) - 1):
        sim = cosine_similarity(blocks[i]["embedding"], blocks[i + 1]["embedding"])
        blocks[i]["next_similarity"] = sim
    if blocks:
        blocks[-1]["next_similarity"] = None
    return blocks


def detect_topic_shift(blocks, threshold=0.90):
    for i in range(len(blocks) - 1):
        sim = blocks[i]["next_similarity"]
        blocks[i]["topic_shift_after"] = sim is not None and sim < threshold
    if blocks:
        blocks[-1]["topic_shift_after"] = True
    return blocks


def merge_blocks_to_topics(blocks):
    topics = []
    topic_id = 0
    current_topic = {
        "topic_id": topic_id,
        "start_time": None,
        "end_time": None,
        "block_ids": [],
        "texts": [],
    }

    for block in blocks:
        if current_topic["start_time"] is None:
            current_topic["start_time"] = block["start_time"]
        current_topic["end_time"] = block["end_time"]
        current_topic["block_ids"].append(block["block_id"])
        current_topic["texts"].append(block["text"])

        if block.get("topic_shift_after", False):
            current_topic["text"] = " ".join(current_topic["texts"])
            del current_topic["texts"]
            topics.append(current_topic)

            topic_id += 1
            current_topic = {
                "topic_id": topic_id,
                "start_time": None,
                "end_time": None,
                "block_ids": [],
                "texts": [],
            }
    return topics


blocks = compute_adjacent_similarity(blocks)
blocks = detect_topic_shift(blocks, threshold=TOPIC_SHIFT_THRESHOLD)
topics = merge_blocks_to_topics(blocks)

print("Total topics:", len(topics))

In [ ]:
block_table = pd.DataFrame([
    {
        "block_id": b["block_id"],
        "timestamp": f'{format_time(b["start_time"])} - {format_time(b["end_time"])}',
        "next_similarity": b["next_similarity"],
        "topic_shift_after": b["topic_shift_after"],
        "text_preview": b["text"][:220],
    }
    for b in blocks
])
display(block_table)

topic_table = pd.DataFrame([
    {
        "topic_id": t["topic_id"],
        "timestamp": f'{format_time(t["start_time"])} - {format_time(t["end_time"])}',
        "block_ids": t["block_ids"],
        "text_preview": t["text"][:350],
        "chars": len(t["text"]),
    }
    for t in topics
])
display(topic_table)

with open(OUTPUT_DIR / "blocks_phobert.json", "w", encoding="utf-8") as f:
    json.dump(blocks, f, ensure_ascii=False, indent=2)

with open(OUTPUT_DIR / "topics_phobert.json", "w", encoding="utf-8") as f:
    json.dump(topics, f, ensure_ascii=False, indent=2)

## 6. Trực quan hóa phân bổ từ vựng và độ dài văn bản

Cell này dùng lại `blocks` và `topics` đã tạo ở trên để xem: phân bổ độ dài block/topic, số từ khác nhau, độ đa dạng từ vựng và các từ xuất hiện nhiều nhất.

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

VI_STOPWORDS = {
    "và", "là", "của", "có", "cho", "trong", "một", "các", "những", "được",
    "với", "thì", "để", "này", "đó", "khi", "ra", "về", "như", "từ", "ở",
    "trên", "bạn", "mình", "tôi", "chúng", "ta", "sẽ", "đã", "đang", "không",
    "nó", "rất", "cũng", "nên", "hay", "lại", "vì", "nếu", "the", "a", "an",
}

def simple_vietnamese_tokens(text):
    text = str(text).lower()
    return re.findall(r"[a-zA-ZÀ-ỹ0-9_]+", text)

def build_text_stats(items, id_col):
    rows = []
    for item in items:
        text = item.get("text", "")
        tokens = simple_vietnamese_tokens(text)
        unique_tokens = set(tokens)
        rows.append({
            id_col: item.get(id_col),
            "chars": len(text),
            "words": len(tokens),
            "unique_words": len(unique_tokens),
            "lexical_diversity": len(unique_tokens) / len(tokens) if tokens else 0,
            "duration_sec": item.get("end_time", 0) - item.get("start_time", 0),
        })
    return pd.DataFrame(rows)

block_stats = build_text_stats(blocks, "block_id")
topic_stats = build_text_stats(topics, "topic_id")

all_tokens = [
    token
    for block in blocks
    for token in simple_vietnamese_tokens(block.get("text", ""))
    if token not in VI_STOPWORDS and len(token) > 1
]
vocab_counts = pd.DataFrame(Counter(all_tokens).most_common(30), columns=["word", "count"])

display(block_stats.describe().round(2))
display(topic_stats.describe().round(2))
display(vocab_counts.head(20))

plt.style.use("seaborn-v0_8-whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].hist(block_stats["words"], bins=20, color="#2f80ed", edgecolor="white")
axes[0, 0].set_title("Phân bổ độ dài block")
axes[0, 0].set_xlabel("Số từ / block")
axes[0, 0].set_ylabel("Số block")

axes[0, 1].bar(topic_stats["topic_id"].astype(str), topic_stats["words"], color="#27ae60")
axes[0, 1].set_title("Độ dài văn bản theo topic")
axes[0, 1].set_xlabel("Topic ID")
axes[0, 1].set_ylabel("Số từ")
axes[0, 1].tick_params(axis="x", rotation=45)

axes[1, 0].plot(block_stats["block_id"], block_stats["unique_words"], marker="o", linewidth=1.5, color="#9b51e0")
axes[1, 0].set_title("Số từ vựng khác nhau theo block")
axes[1, 0].set_xlabel("Block ID")
axes[1, 0].set_ylabel("Unique words")

axes[1, 1].barh(vocab_counts["word"].iloc[::-1], vocab_counts["count"].iloc[::-1], color="#f2994a")
axes[1, 1].set_title("Top 30 từ xuất hiện nhiều nhất")
axes[1, 1].set_xlabel("Tần suất")

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(topic_stats["topic_id"], topic_stats["lexical_diversity"], marker="o", color="#eb5757")
ax.set_title("Độ đa dạng từ vựng theo topic")
ax.set_xlabel("Topic ID")
ax.set_ylabel("Unique words / Total words")
ax.set_ylim(0, min(1, topic_stats["lexical_diversity"].max() + 0.1))
plt.tight_layout()
plt.show()

## 7. Load Qwen2.5 để tóm tắt

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

summary_tokenizer = AutoTokenizer.from_pretrained(SUMMARY_MODEL, trust_remote_code=True)
summary_model = AutoModelForCausalLM.from_pretrained(
    SUMMARY_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
summary_model.eval()
print("Loaded summary model:", SUMMARY_MODEL)

In [ ]:
def summary_generate(system_prompt, user_prompt, max_new_tokens=700, temperature=0.25):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    text = summary_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = summary_tokenizer(text, return_tensors="pt", truncation=True, max_length=12000).to(summary_model.device)

    with torch.no_grad():
        outputs = summary_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.85,
            repetition_penalty=1.05,
        )
    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    return summary_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

## 8. Tóm tắt từng topic và tóm tắt toàn video

In [ ]:
def summarize_single_topic(topic):
    system_prompt = """
Ban la tro ly AI chuyen tom tat video bai giang IT/AI bang tieng Viet.
Nhiem vu cua ban la doc transcript cua mot topic va tao tieu de + bullet points ngan gon, chinh xac.
Khong bia noi dung ngoai transcript.
Tra loi bang tieng Viet.
"""
    user_prompt = f"""
Day la transcript cua mot topic trong video.

Timestamp: {format_time(topic['start_time'])} - {format_time(topic['end_time'])}

Transcript:
{topic['text']}

Hay tao ket qua theo dung format sau:

TITLE: <tieu de ngan gon cho topic>

BULLET_POINTS:
- <y chinh 1>
- <y chinh 2>
- <y chinh 3>
- <y chinh 4 neu co>
"""
    return summary_generate(system_prompt, user_prompt, max_new_tokens=500)


for idx, topic in enumerate(topics):
    print(f"Summarizing topic {idx + 1}/{len(topics)}...")
    topic["timestamp"] = f'{format_time(topic["start_time"])} - {format_time(topic["end_time"])}'
    topic["summary_raw"] = summarize_single_topic(topic)

with open(OUTPUT_DIR / "topics_phobert_qwen_summary.json", "w", encoding="utf-8") as f:
    json.dump(topics, f, ensure_ascii=False, indent=2)

for topic in topics:
    display(Markdown(f"### Topic {topic['topic_id']} | {topic['timestamp']}\n\n{topic['summary_raw']}"))

In [ ]:
def generate_final_summary(topic_list):
    topic_context = "\n".join([
        f"""
Topic {topic['topic_id']}
Timestamp: {topic.get('timestamp', '')}
{topic.get('summary_raw', '')}
"""
        for topic in topic_list
    ])

    system_prompt = """
Ban la tro ly AI chuyen tom tat video bai giang IT/AI bang tieng Viet.
Ban can tong hop cac topic summaries thanh mot ban tom tat cuoi cung ro rang, co cau truc.
Khong bia them noi dung khong co trong du lieu.
"""
    user_prompt = f"""
Duoi day la danh sach topic summaries cua mot video:

{topic_context}

Hay tao final summary theo dung format sau:

FINAL SUMMARY

1. Tom tat ngan 5-6 cau:
<viet doan tom tat ngan>

2. Cac noi dung chinh:
- <noi dung chinh 1>
- <noi dung chinh 2>
- <noi dung chinh 3>
- <noi dung chinh 4>
- <noi dung chinh 5 neu co>

3. Dien tien noi dung theo timestamp:
- <timestamp>: <noi dung>
- <timestamp>: <noi dung>
- <timestamp>: <noi dung>

4. Ket luan:
<ket luan ngan ve noi dung video>
"""
    return summary_generate(system_prompt, user_prompt, max_new_tokens=1200)


final_summary = generate_final_summary(topics)
(OUTPUT_DIR / "final_summary_phobert_qwen.txt").write_text(final_summary, encoding="utf-8")
display(Markdown(final_summary))

## 9. Nhập câu hỏi và hỏi đáp RAG

In [ ]:
# QA dùng Qwen3-4B-Instruct-2507, tách riêng với model tóm tắt.
qa_tokenizer = AutoTokenizer.from_pretrained(QA_MODEL, trust_remote_code=True)
qa_model = AutoModelForCausalLM.from_pretrained(
    QA_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
qa_model.eval()
print("Loaded QA model:", QA_MODEL)


def qa_generate(system_prompt, user_prompt, max_new_tokens=800, temperature=0.2):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    text = qa_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = qa_tokenizer(text, return_tensors="pt", truncation=True, max_length=12000).to(qa_model.device)

    with torch.no_grad():
        outputs = qa_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.8,
            repetition_penalty=1.05,
        )
    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    return qa_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


def search_relevant_blocks_phobert(question, blocks, top_k=5):
    question_emb = get_phobert_embedding(question)
    results = []
    for block in blocks:
        score = cosine_similarity(question_emb, block["embedding"])
        results.append({
            "block_id": block["block_id"],
            "start_time": block["start_time"],
            "end_time": block["end_time"],
            "text": block["text"],
            "score": score,
        })
    return sorted(results, key=lambda x: x["score"], reverse=True)[:top_k]


def build_metadata_context(metadata):
    description = metadata.get("description", "") or ""
    if len(description) > 1500:
        description = description[:1500] + "..."
    return f"""
VIDEO METADATA:
Title: {metadata.get('title') or 'Khong ro'}
Channel: {metadata.get('channel_name') or metadata.get('uploader') or 'Khong ro'}
URL: {metadata.get('webpage_url') or metadata.get('youtube_url') or 'Khong ro'}
Duration: {metadata.get('total_duration') or 'Khong ro'} seconds
Upload date: {metadata.get('published_at') or 'Khong ro'}
View count: {metadata.get('view_count', 'Khong ro')}
Like count: {metadata.get('like_count', 'Khong ro')}

Description:
{description}
"""


def build_rag_context(retrieved_blocks):
    parts = []
    for item in retrieved_blocks:
        parts.append(f"""
[Block {item['block_id']}]
Timestamp: {format_time(item['start_time'])} - {format_time(item['end_time'])}
Similarity score: {item['score']:.4f}
Content:
{item['text']}
""")
    return "\n".join(parts)


def answer_question_rag_phobert(question, blocks, metadata, top_k=5):
    retrieved_blocks = search_relevant_blocks_phobert(question, blocks, top_k=top_k)
    metadata_context = build_metadata_context(metadata)
    rag_context = build_rag_context(retrieved_blocks)

    system_prompt = """
Ban la tro ly hoi dap video bai giang IT/AI.
Ban chi duoc tra loi dua tren VIDEO METADATA va TRANSCRIPT CONTEXT duoc cung cap.
Neu cau hoi hoi ve thong tin chung cua video nhu tieu de, kenh, URL, thoi luong, hay uu tien dung VIDEO METADATA.
Neu cau hoi hoi ve noi dung bai giang, hay dung TRANSCRIPT CONTEXT.
Neu ca metadata va transcript deu khong du thong tin, hay noi ro: "Khong tim thay du thong tin trong du lieu video."
Luon tra loi bang tieng Viet.
Luon kem timestamp khi cau tra loi dua tren transcript.
Khong bia noi dung ngoai du lieu duoc cung cap.
"""

    user_prompt = f"""
QUESTION:
{question}

EMBEDDING MODEL:
vinai/phobert-base

{metadata_context}

TRANSCRIPT CONTEXT:
{rag_context}

Hay tra loi theo format:

CAU TRA LOI:
<tra loi ro rang, dung trong tam>

TIMESTAMP LIEN QUAN:
- <mm:ss - mm:ss>: <vi sao doan nay lien quan>
Neu cau hoi chi lien quan den metadata, ghi: "Khong can timestamp vi cau hoi dua tren metadata."

BLOCK DUOC TRUY XUAT:
- Block <id>, score=<score>
"""

    answer = qa_generate(system_prompt, user_prompt, max_new_tokens=800, temperature=0.2)
    return {
        "question": question,
        "answer": answer,
        "retrieved_blocks": retrieved_blocks,
    }

In [ ]:
import itertools
import sys
import threading


def run_with_thinking_effect(fn, *args, **kwargs):
    result_box = {}
    error_box = {}
    done = threading.Event()

    def worker():
        try:
            result_box["value"] = fn(*args, **kwargs)
        except Exception as exc:
            error_box["error"] = exc
        finally:
            done.set()

    start_time = time.time()
    thread = threading.Thread(target=worker)
    thread.start()

    frames = ["Thinking   ", "Thinking.  ", "Thinking.. ", "Thinking..."]
    for frame in itertools.cycle(frames):
        if done.is_set():
            break
        elapsed = time.time() - start_time
        print(f"\r{frame} | đang tạo câu trả lời: {elapsed:.1f}s", end="", flush=True)
        time.sleep(0.4)

    thread.join()
    elapsed = time.time() - start_time
    print(f"\rHoàn tất phản hồi trong {elapsed:.2f}s" + " " * 20)

    if "error" in error_box:
        raise error_box["error"]

    return result_box["value"], elapsed


qa_history = []

print("Nhập câu hỏi về video. Gõ 'end' để dừng.")

while True:
    question = input("Câu hỏi: ").strip()

    if question.lower() == "end":
        print("Đã dừng hỏi đáp.")
        break

    if not question:
        print("Bạn chưa nhập câu hỏi. Gõ lại hoặc nhập 'end' để dừng.")
        continue

    qa_result, response_time = run_with_thinking_effect(
        answer_question_rag_phobert,
        question,
        blocks,
        metadata,
        top_k=TOP_K,
    )
    qa_result["response_time_seconds"] = round(response_time, 2)
    qa_history.append(qa_result)

    display(Markdown(
        f"## Câu hỏi\n{question}\n\n"
        f"## Câu trả lời\n{qa_result['answer']}\n\n"
        f"**Thời gian phản hồi:** {response_time:.2f} giây"
    ))

    retrieved_table = pd.DataFrame([
        {
            "block_id": b["block_id"],
            "timestamp": f'{format_time(b["start_time"])} - {format_time(b["end_time"])}',
            "score": b["score"],
            "text_preview": b["text"][:350],
        }
        for b in qa_result["retrieved_blocks"]
    ])
    display(retrieved_table)

    with open(OUTPUT_DIR / "qa_phobert_qwen_history.json", "w", encoding="utf-8") as f:
        json.dump(qa_history, f, ensure_ascii=False, indent=2)

print("Tổng số câu hỏi đã lưu:", len(qa_history))